# 01 - Parse and inspect one calculation

This notebook follows the same `water_mp2.out` example as the quick start. It introduces the `batch -> file -> frame` access pattern and shows which results are available in the source. Download the example from the documentation and save it as `water_mp2.out` before running the notebook.

In [1]:
from pathlib import Path

from molop import AutoParser, molopconfig

molopconfig.quiet()
sample_candidates = (
    Path("docs/assets/examples/water_mp2.out"),
    Path("../../assets/examples/water_mp2.out"),
    Path("water_mp2.out"),
)
sample_path = next((path for path in sample_candidates if path.is_file()), None)
if sample_path is None:
    raise FileNotFoundError("Place water_mp2.out beside the notebook or use the bundled example.")

batch = AutoParser(sample_path, n_jobs=1)
parsed_file = batch[0]
frame = parsed_file[-1]

print("files:", len(batch))
print("frames:", len(parsed_file))
print("format:", parsed_file.detected_format_id)
print("atoms and coordinates:", len(frame.atoms), frame.coords.shape)
print("energy (hartree):", frame.energies.total_energy.m_as("hartree"))

files: 1
frames: 1
format: orcaout
atoms and coordinates: 3 (3, 3)
energy (hartree): -74.999374598107


The parser identifies the source as ORCA output and exposes one final frame in this example.

In [2]:
import pandas as pd
from IPython.display import display

summary = batch.to_summary_df(
    frame=-1,
    brief=False,
    flatten_columns=True,
)
with pd.option_context("display.max_columns", None, "display.max_rows", None, "display.width", 240):
    display(summary)

,DiskStorage.FilePath,DiskStorage.FileFormat,General.Charge,General.Multiplicity,General.CanonicalSMILES,General.NumAtoms,General.FrameID,Calc Parameter.Software,Calc Parameter.Version,Calc Parameter.Method,Calc Parameter.BasisSet,Calc Parameter.Functional,Calc Parameter.Keywords,Environment.SolventModel,Environment.Solvent,Status.IsError,Status.IsNormal,Status.IsTS,Status.IsOptimized,Energy.electronic_energy.hartree,Energy.reference_energy.hartree,Energy.mp2_energy.hartree,Energy.total_energy.hartree
0,/home/tmj/proj/MolOP/docs/assets/examples/wate...,.out,0,1,O,3,0,ORCA,6.0.1,MP2,sto-3g,,MP2 sto-3g,None,None,False,True,False,False,-74.999375,-74.963574,-74.999375,-74.999375


Scientific result containers are optional. Check a container before using its fields.

In [3]:
if frame.energies and frame.energies.total_energy is not None:
    print(frame.energies.total_energy.m_as("eV"))

print("vibrations available:", frame.vibrations is not None)

-2040.8369503966658
vibrations available: False


Accessing `rdmol` provides a molecular graph and records the topology reconstruction status.

In [4]:
mol = frame.rdmol
print("bonds:", mol.GetNumBonds())
print("topology status:", frame.topology_reconstruction_status)

bonds: 2
topology status: succeeded
